# COVER-KBC v2 — Colab executionThin driver for the **AKBC Shared Task 2026** system. Every cell calls repositorycode; no COVER logic is reimplemented here.**Frozen architecture**| role | model | published params ||---|---|---|| enumerator | `mistralai/Mistral-Small-3.2-24B-Instruct-2506` | 24,011,361,280 || verifier | `Qwen/Qwen3.5-4B` | 4,659,865,088 || **total** | | **28,671,226,368** (28.67B ≤ 32B ✅) |Execution is **staged**, so the GPU never has to hold both models at once:```PHASE A  load Mistral -> enumerate -> persist graphs -> unloadPHASE B  load Qwen    -> cross-model recall + verify -> persist -> unloadPHASE C  no model     -> RCSE, controller, selection -> predictions -> evaluate```The counted parameter budget is unchanged by the split: the challenge countspublished parameters used at inference, not peak VRAM.**Runtime:** set *Runtime → Change runtime type → GPU* (A100 / L4 preferred).

## 1. Clone the repository

In [ ]:
REPO_URL = "https://github.com/YOUR_ORG/FactElicit-AKBC.git"  # <- set meCOMMIT   = ""   # optional: pin an exact commit for reproducibilityimport os, subprocess, sysif not os.path.isdir("FactElicit-AKBC"):    subprocess.run(["git", "clone", REPO_URL], check=True)os.chdir("/content/FactElicit-AKBC")if COMMIT:    subprocess.run(["git", "checkout", COMMIT], check=True)print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 2. Install dependencies

In [ ]:
!pip -q install -e '.[hf]'!pip -q install bitsandbytesimport torch, transformersprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print("gpu", p.name, round(p.total_memory / 2**30, 1), "GiB")print("transformers", transformers.__version__)

## 3. Optional: Hugging Face auth and DriveMistral-Small may require accepting its licence on the Hub. Mounting Drive keepsthe model cache and run artifacts across sessions.

In [ ]:
USE_HF_TOKEN = FalseUSE_DRIVE    = Falseif USE_HF_TOKEN:    from huggingface_hub import login    from google.colab import userdata    login(userdata.get("HF_TOKEN"))if USE_DRIVE:    from google.colab import drive    drive.mount("/content/drive")    import os    os.environ["HF_HOME"] = "/content/drive/MyDrive/cover_kbc/hf-cache"    os.makedirs(os.environ["HF_HOME"], exist_ok=True)    print("HF_HOME =", os.environ["HF_HOME"])

## 4. Verify benchmark integrityThe official snapshot must be untouched before anything else runs.

In [ ]:
import subprocessdirty = subprocess.run(["git", "status", "--porcelain", "benchmark/"],                       capture_output=True, text=True).stdout.strip()assert not dirty, f"benchmark/ has been modified:\n{dirty}"print("benchmark/ is clean ✅")!python -c "import _bootstrap" 2>/dev/null || true!PYTHONPATH=src python -c "\from cover_kbc.evaluation.official import evaluator_checksum; \from cover_kbc.data.loader import load_all_splits; \print('evaluator sha256:', evaluator_checksum()); \[print(f'{k:6s} {len(v):4d} rows  {v.sha256[:16]}') for k, v in load_all_splits().items()]"

## 5. Verify the 32B parameter budgetFails closed: unverified provenance is not a pass.

In [ ]:
CONFIG = "configs/experiments/cover_kbc_v2_mistral24_qwen4.yaml"!python scripts/audit_model_budget.py {CONFIG}

## 6. Configure the runStart with a small `LIMIT` to smoke-test the whole path cheaply, then set`LIMIT = 0` for the full split. The same code path scales to the blind testsplit.

In [ ]:
SPLIT = "val"   # "train" | "val" | "test" (test is blind: no local scoring)LIMIT = 20      # 0 = the whole splitRUN_DIR = ""    # set to resume an existing run directorylimit_arg = f"--limit {LIMIT}" if LIMIT else ""print(f"config={CONFIG}\nsplit={SPLIT}\nlimit={LIMIT or 'all'}")

## 7. PHASE A — enumerate (Mistral 24B)Loads the enumerator, discovers candidates, persists the evidence graphs, then the model can be released.

In [ ]:
!python scripts/run_staged.py enumerate --config {CONFIG} --split {SPLIT} {limit_arg}

In [ ]:
# Pick up the run directory the enumerate phase created.import pathlibif not RUN_DIR:    RUN_DIR = str(sorted(pathlib.Path("outputs").glob("*"), key=lambda p: p.stat().st_mtime)[-1])print("RUN_DIR =", RUN_DIR)

In [ ]:
# Free the enumerator before loading the verifier.import gc, torchgc.collect()if torch.cuda.is_available():    torch.cuda.empty_cache()    print("VRAM allocated: %.2f GiB" % (torch.cuda.memory_allocated() / 2**30))

## 8. PHASE B — cross-model recall + blind verification (Qwen 4.66B)The verifier both *independently recalls* objects (genuine cross-model evidence) and scores shown candidates with calibrated VALID/INVALID/UNKNOWN logits.

In [ ]:
!python scripts/run_staged.py verify --config {CONFIG} --run-dir {RUN_DIR}

## 9. PHASE C — control, selection, evaluation (no model)Entirely non-neural, so it is cheap to re-run with different thresholds against one expensive set of generations.

In [ ]:
!python scripts/run_staged.py decide --config {CONFIG} --split {SPLIT} --run-dir {RUN_DIR}

## 10. Official evaluatorRun `benchmark/evaluate.py` exactly as a participant would, as an independent check of the in-process numbers.

In [ ]:
if SPLIT != "test":    !python benchmark/evaluate.py -p {RUN_DIR}/predictions.jsonl -g benchmark/data/{SPLIT}.jsonlelse:    print("test is the blind split - no gold objects, so it cannot be scored locally.")

## 11. Inspect the artifacts

In [ ]:
import json, pathlibrun = pathlib.Path(RUN_DIR)for name in sorted(p.name for p in run.iterdir()):    print(f"  {name:32s} {(run / name).stat().st_size:>12,} bytes")diag = run / "diagnostics.json"if diag.is_file():    print()    print(json.dumps(json.loads(diag.read_text()), indent=2))

In [ ]:
# One worked example: candidates, evidence provenance, and the S(o) breakdown.import json, pathlibtrace = pathlib.Path(RUN_DIR) / "trace.jsonl"row = json.loads(trace.read_text().splitlines()[0])print(row["SubjectEntity"], "|", row["Relation"], "->", row["ObjectEntities"])print("empty_reason:", row["empty_reason"], "| stopped:", row["stopped_reason"])for c in row["candidates"][:5]:    print(f"  {c['display_value']:28s} tier={c['tier']:20s} indep={c['independent_support']} "          f"facets={c['num_facets']} S={c['score']:.3f}")    print(f"      {c['score_breakdown']}")

## 12. Save resultsCopy artifacts to Drive so they survive the session, then commit metrics back tothe repository if desired.

In [ ]:
if USE_DRIVE:    import shutil, pathlib    dest = pathlib.Path("/content/drive/MyDrive/cover_kbc/runs") / pathlib.Path(RUN_DIR).name    shutil.copytree(RUN_DIR, dest, dirs_exist_ok=True)    print("saved to", dest)

---### Notes* **Ablations.** Copy the config and toggle `pipeline.enable_verifier`,  `enable_calibrated_gate`, `enable_active_controller`,  `enable_cross_model_recall`, `use_calibration`. Phase C alone can be re-run for  anything that only changes thresholds.* **Threshold calibration.** Tune on `train`, freeze the config, *then* score  `val`. Tuning on val and reporting that same val number is not a measurement.* **Closed book.** No cell may add web search, RAG, or a KB lookup to the  prediction path. Model downloads are setup, not inference-time retrieval.